Need to make a figure (or multiple figures) of the Li-polluted WD spectra. Nominally this for the lightning talk, but it will probably become part of the real paper too. Or will be recycled. I don't know.

To that end, I'm going to pull code from "multiple_object_spectra.ipynb" since this is basically the same thing, but I don't want to try to clean up that whole mess.

In [1]:
from __future__ import print_function



import matplotlib
matplotlib.use('pdf')



import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
start = time.time()

import spec_plot_tools as spt
import cal_params as cp

import plot_spec as ps

print(os.getcwd())

all_wctb
/Users/BenKaiser/Desktop/radial_velocity_calculations


In [2]:
target_dir='/Users/BenKaiser/Desktop/400_both/special_lithium/'

In [3]:
lhs_400m1_file='ravg_fwctb.LHS2534_400m1.fits'
lhs_400m2_file='ravg_fwctb.LHS2534_400m2.fits'
J1824_400m1_file='ravg_fwctb.WDJ1824p1213actual_400m1.fits'
J1824_400m2_file='ravg_fwctb.WDJ1824p1213actual_400m2.fits'
J2317_400m2_file='ravg_fwctb.WDJ2317p1830_400m2.fits'
#There isn't a 400M1 file for J2317 because we never took one in an attempt to maximize SNR in 400M2

In [4]:
os.chdir(target_dir)

In [5]:
figure_output_dir='/Users/BenKaiser/Desktop/'

In [6]:
range_400m1=[3600,6660]
range_400m2=[6660,10000]

In [7]:
lhs_400m1_spec, lhs_400m1_header, noise=spt.retrieve_spec(lhs_400m1_file)
lhs_400m2_spec, lhs_400m2_header, noise=spt.retrieve_spec(lhs_400m2_file)
J1824_400m1_spec, J1824_400m1_header, noise=spt.retrieve_spec(J1824_400m1_file)
J1824_400m2_spec, J1824_400m2_header, noise=spt.retrieve_spec(J1824_400m2_file)
J2317_400m2_spec, J2317_400m2_header, noise=spt.retrieve_spec(J2317_400m2_file)

In [8]:
sm_lhs_400m1=ps.convolve_spectrum(lhs_400m1_spec, lhs_400m1_header, kernel_type='box', pix_width=3)
sm_lhs_400m2=ps.convolve_spectrum(lhs_400m2_spec, lhs_400m2_header, kernel_type='box', pix_width=3)
sm_J1824_400m1=ps.convolve_spectrum(J1824_400m1_spec, J1824_400m1_header, kernel_type='box', pix_width=3)
sm_J1824_400m2=ps.convolve_spectrum(J1824_400m2_spec, J1824_400m2_header, kernel_type='box', pix_width=3)
sm_J2317_400m2=ps.convolve_spectrum(J2317_400m2_spec, J2317_400m2_header, kernel_type='box', pix_width=3)

In [9]:
norm_range=[6630,6690]

In [10]:
norm_lhs_400m1=ps.norm_spectrum(sm_lhs_400m1, norm_range, show_norm_range=False)
norm_lhs_400m2=ps.norm_spectrum(sm_lhs_400m2, norm_range, show_norm_range=False)
norm_J1824_400m1=ps.norm_spectrum(sm_J1824_400m1, norm_range, show_norm_range=False)
norm_J1824_400m2=ps.norm_spectrum(sm_J1824_400m2, norm_range, show_norm_range=False)
norm_J2317_400m2=ps.norm_spectrum(sm_J2317_400m2, norm_range, show_norm_range=False)



wavelength-based norm range selected [6630, 6690]





wavelength-based norm range selected [6630, 6690]





wavelength-based norm range selected [6630, 6690]





wavelength-based norm range selected [6630, 6690]





wavelength-based norm range selected [6630, 6690]





In [11]:
norm_lhs_400m1=spt.clean_spectrum(norm_lhs_400m1, range_400m1[0],range_400m1[1], [])
norm_lhs_400m2=spt.clean_spectrum(norm_lhs_400m2, range_400m2[0],range_400m2[1], [])
norm_J1824_400m1=spt.clean_spectrum(norm_J1824_400m1, range_400m1[0],range_400m1[1], [])
norm_J1824_400m2=spt.clean_spectrum(norm_J1824_400m2,  range_400m2[0],range_400m2[1], [])


In [12]:
#plt.rc('font',size=12)
spt.initiate_science_plot()
#fig= plt.figure(figsize=(10,6))
fig=spt.start_ApJ_fig(width_cols=2,width_height=[1.,6./10.])
spt.show_plot(show_legend=False,actually_show=False,show_label=False,line_id='cool_wd',convert_to_air=True)

label_pos=3.3
label_off=20
#x_pos= 6000
#x_pos= 8000
x_pos= 6750
y_pos= 1.1

plt.plot(norm_lhs_400m1[0], norm_lhs_400m1[1]+2, color='k')
plt.plot(norm_lhs_400m2[0], norm_lhs_400m2[1]+2, color='k')
plt.plot(norm_J1824_400m1[0], norm_J1824_400m1[1], color='k')
plt.plot(norm_J1824_400m2[0], norm_J1824_400m2[1], color='k')
plt.plot(norm_J2317_400m2[0], norm_J2317_400m2[1]+1, color='k')



plt.text(x_pos, y_pos+2, 'LHS 2534', color='k')
plt.text(x_pos, y_pos, 'WD J1824+1213', color='k')
plt.text(x_pos, y_pos+1, 'WD J2317+1830', color='k')

plt.ylabel(r'Flux ($f_{\lambda}$ arbitrary units)')
#plt.xlabel(r'$\lambda(\AA)$')
plt.xlabel(r'Wavelength $(\mathrm{\AA})$')



legend_lines=['Li','Na','MgH','K','Ca']
for element in legend_lines:
    plt.axvline(x=-1000.,linestyle='--',color=cp.line_color_dict[element],label=element)

plt.xlim(3750,9000)
plt.ylim(-0.4,3.5)
plt.legend(loc='lower right',framealpha=1)
#spt.show_plot(show_legend=False,actually_show=False,show_label=False,line_id='cool_wd',convert_to_air=True)







print(os.getcwd())
os.chdir(figure_output_dir)
print(os.getcwd())
start = time.time()
print(start)
time_string=str(start).split('.')[0]
plt.savefig('multiple_LiWD_spectra_'+time_string+'.pdf')#plt.grid(True)



spt.show_plot(show_legend=False)

/Users/BenKaiser/Desktop/Goodman_ref_files/line_lists/all_lines_cool_WD.csv
wavelengths.min:  5891.583264
new_wavelengths.min(): 5889.950866765008
wavelengths.min:  5897.558147
new_wavelengths.min(): 5895.924149766943
wavelengths.min:  6709.61
new_wavelengths.min(): 6707.758043628569
wavelengths.min:  6709.76
new_wavelengths.min(): 6707.908003287872
wavelengths.min:  7667.008906
new_wavelengths.min(): 7664.899016001414
wavelengths.min:  7701.083536
new_wavelengths.min(): 7698.9644515250075
wavelengths.min:  8185.5054
new_wavelengths.min(): 8183.255515614788
wavelengths.min:  8197.0434
new_wavelengths.min(): 8194.790398371782
wavelengths.min:  8197.0766
new_wavelengths.min(): 8194.82358940196
wavelengths.min:  3969.59
new_wavelengths.min(): 3968.4672118153667
wavelengths.min:  3934.77
new_wavelengths.min(): 3933.6562946887625
wavelengths.min:  4227.92
new_wavelengths.min(): 4226.729580953195
wavelengths.min:  8500.35
new_wavelengths.min(): 8498.015025790284
wavelengths.min:  8544.44
new

/Users/BenKaiser/Desktop/radial_velocity_calculations/spec_plot_tools.py:501: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [13]:
plt.rc('font',size=12)
fig= plt.figure(figsize=(10,6))
plt.plot(norm_J1824_400m1[0], norm_J1824_400m1[1], color='k')
plt.plot(norm_J1824_400m2[0], norm_J1824_400m2[1], color='k')
plt.text(x_pos, y_pos, 'WD J1824+1213', color='k')
plt.ylabel(r'Flux ($f_{\lambda}$ arbitrary units)')
plt.xlabel(r'Wavelength $(\AA)$')
plt.xlim(3750,9000)


spt.show_plot(show_legend=False)


/Users/BenKaiser/Desktop/radial_velocity_calculations/spec_plot_tools.py:501: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [14]:
plt.rc('font',size=12)
fig= plt.figure(figsize=(10,6))
plt.plot(norm_J1824_400m1[0], norm_J1824_400m1[1], color='k')
plt.plot(norm_J1824_400m2[0], norm_J1824_400m2[1], color='k')
plt.text(x_pos, y_pos, 'WD J1824+1213', color='k')
plt.ylabel(r'Flux ($f_{\lambda}$ arbitrary units)')
plt.xlabel(r'Wavelength $(\AA)$')
plt.xlim(7700,8000)


spt.show_plot(show_legend=False)

/Users/BenKaiser/Desktop/radial_velocity_calculations/spec_plot_tools.py:501: UserWarning: Matplotlib is currently using pdf, which is a non-GUI backend, so cannot show the figure.
  plt.show()
